# 第 6 周练习 —— 训练规模重要吗？（mugisha_caleb_didier）

## 练习目标（理念）

课程里提到：OpenAI 常建议用大约 **50–100** 条样本做微调。但当你把训练集从 50 一路加到 5000，**MAE 到底怎么变？**

本笔记本对 `GPT-4.1-nano` 分别用 **50 / 100 / 200 / 400 / 1000 / 2000 / 5000** 条训练样本各开一个微调作业，再画学习曲线。7 个作业在 OpenAI 上**并行**跑，等待时间大致等于只跑一个。

## 和本课的关系

- Day 3：随机 / 均值基线（先有下限参照）
- Day 4：零样本前沿模型（学习曲线上的「0 条训练」点）
- Day 5：微调 JSONL + `fine_tuning.jobs` + 评估

## 数据集

`ed-donner/items_lite`：约 20K 亚马逊商品，带 `summary` + `price`（约 $0.50–$999）。

## 怎么跑

1. `.env` 配好 `OPENAI_API_KEY`（`load_dotenv(override=True)`）
2. 自上而下运行；微调格会花钱、要等，可隔几分钟重跑「监控作业」那一格
3. 全部 `succeeded` 后再跑评估与绘图


In [ ]:
# ========== 导入：微调实验要用到的工具箱 ==========

# os：读环境变量；re：从模型输出里抠数字；io：内存里构造上传文件
import os
import re
import io
import json
import random
# numpy / matplotlib：算 MAE、画学习曲线
import numpy as np
import matplotlib.pyplot as plt
# load_dotenv：把 .env 密钥读进环境，避免写进代码
from dotenv import load_dotenv
# load_dataset：从 HuggingFace Hub 拉 items_lite
from datasets import load_dataset
# OpenAI SDK：Chat Completions + Files + Fine-tuning Jobs
from openai import OpenAI
# tqdm.notebook：在 Jupyter 里显示进度条
from tqdm.notebook import tqdm


In [ ]:
# ========== 环境 + 客户端 + 数据集 ==========

# override=True：用 .env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 默认读 OPENAI_API_KEY，创建官方客户端
client = OpenAI()

# 拉 lite 数据集；后面用 train / test（validation 在微调格再用）
ds = load_dataset("ed-donner/items_lite")
train = ds["train"]
test = ds["test"]

# 打印规模与一条样例，确认字段：title / price / summary
print(f"Train: {len(train):,} items | Test: {len(test):,} items")
print(f"\nSample item:")
print(f"  Title: {train[0]['title']}")
print(f"  Price: ${train[0]['price']:.2f}")
print(f"  Summary: {train[0]['summary'][:200]}...")


## 评估助手

下面两格：`post_process` 把模型胡写的字符串收成合理美元价；`evaluate` 在测试集上跑 n 条并算 MAE。


In [ ]:
# ========== 后处理：从 LLM 输出里提取数字价格 ==========

def post_process(value):
    # 若是字符串：去掉 $ 与千分位逗号，再用正则抓第一个数字
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", value)
        if not match:
            # 解析失败：回退 0（后面会被 clip 到下限 1.0）
            return 0
        price = float(match.group())
    else:
        # 已经是数值（基线直接返回 float/int）则直接转 float
        price = float(value)
    
    # 裁剪到数据集实际价格区间 [1, 999]
    price = max(1.0, min(999.0, price))
    
    # 对齐常见零售尾数：.00 / .49 / .95 / .99（更像货架价）
    endings = [0.00, 0.49, 0.95, 0.99]
    base = int(price)
    best = min(endings, key=lambda e: abs(price - (base + e)))
    return round(base + best, 2)


In [ ]:
# ========== 彩色误差打印 + evaluate：统一评测入口 ==========

# ANSI 颜色：绿 / 黄 / 红，终端里一眼看出误差好坏
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

def color_for(error, truth):
    # 绝对误差小，或相对误差 <20% → 绿
    if error < 40 or error / truth < 0.2:
        return GREEN
    # 中等误差 → 黄
    elif error < 80 or error / truth < 0.4:
        return YELLOW
    # 否则红
    return RED


def evaluate(predictor, dataset, n=200):
    """Run predictor on n test items. Returns (predictions, actuals, mae)."""
    preds, actuals = [], []
    for i in tqdm(range(n)):
        item = dataset[i]
        # 预测后一律走 post_process，保证基线与 LLM 输出同一套数值规则
        guess = post_process(predictor(item))
        truth = item["price"]
        error = abs(guess - truth)
        c = color_for(error, truth)
        # 不换行连打彩色 $|error|，形成一条误差色带
        print(f"{c}${error:.0f}{RESET} ", end="")
        preds.append(guess)
        actuals.append(truth)
    # MAE = 绝对误差均值
    mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
    print(f"\n\nMAE: ${mae:.2f}")
    return preds, actuals, mae


## 基线

先立两根「下限标尺」：随机瞎猜、永远猜训练集均值。后面任何 LLM / 微调若打不过它们，说明没学到有用信号。


In [ ]:
# ========== 随机基线：在 $1–$999 均匀瞎猜 ==========

# 固定种子，保证每次重跑随机基线可复现
random.seed(42)

def random_predictor(item):
    # item 没用到：故意忽略文本，模拟「完全没读描述」
    return random.randrange(1, 1000)

# 只要 MAE；预测序列用 _ 丢掉
_, _, random_mae = evaluate(random_predictor, test)


In [ ]:
# ========== 均值基线：永远预测训练集平均价 ==========

# 先算训练集价格均值（常数预测器）
train_avg = np.mean([item["price"] for item in train])
print(f"Training average price: ${train_avg:.2f}")

def mean_predictor(item):
    # 同样忽略 item，始终返回同一个均值
    return train_avg

_, _, mean_mae = evaluate(mean_predictor, test)


## 零样本 GPT-4.1-nano

这是学习曲线上的「**0 条训练样本**」数据点：同一提示、同一测试集，后面用来和各规模微调对比。


In [ ]:
# ========== 零样本定价：不训练，直接 Chat Completions ==========

def zero_shot(item):
    # 提示词字符串影响行为：保持英文原文
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": prompt}],
        # max_tokens=10：只要短答案；temperature=0：尽量确定性
        max_tokens=10,
        temperature=0
    )
    return response.choices[0].message.content

# 保存预测/真值/MAE，后面画散点图会用到
zero_preds, zero_actuals, zero_mae = evaluate(zero_shot, test)


## 准备并启动全部 7 个微调作业

训练规模：`50, 100, 200, 400, 1000, 2000, 5000`。  
**验证集统一 50 条**（所有作业共用同一份 validation JSONL），变量只有训练集大小。


In [ ]:
# ========== JSONL 构造：chat 格式的 user/assistant 成对样本 ==========

# 七档训练规模（后面启动作业、画曲线都围绕这个列表）
SIZES = [50, 100, 200, 400, 1000, 2000, 5000]

def messages_for(item):
    # user：估价指令 + summary；assistant：带 $ 的真价（监督标签）
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": f"${item['price']:.2f}"}
    ]

def make_jsonl(dataset, n):
    # 取前 n 条，每行一个 {"messages": [...]} JSON
    lines = []
    for i in range(n):
        row = json.dumps({"messages": messages_for(dataset[i])})
        lines.append(row)
    return "\n".join(lines)

# 验证集：Hub 的 validation split；固定 50 条给所有作业共用
val = ds["validation"]
val_jsonl = make_jsonl(val, 50)

print(f"Validation set: 50 items")
# 抽第一行 user content 前 80 字符做目检
print(f"Sample line: {json.loads(val_jsonl.split(chr(10))[0])['messages'][0]['content'][:80]}...")


In [ ]:
# ========== 上传文件并启动全部作业（遇限流则等待重试）==========

import time

# size → job.id 映射，后面监控/取模型名都靠它
jobs = {}

# 共享验证文件只上传一次，所有作业复用同一个 file id
val_file = client.files.create(
    file=("val.jsonl", io.BytesIO(val_jsonl.encode())),
    purpose="fine-tune"
)
print(f"Uploaded validation file: {val_file.id}")

for size in SIZES:
    # 按当前规模从 train 生成 JSONL，并上传为 train_{size}.jsonl
    train_jsonl = make_jsonl(train, size)
    train_file = client.files.create(
        file=(f"train_{size}.jsonl", io.BytesIO(train_jsonl.encode())),
        purpose="fine-tune"
    )
    # OpenAI 对并发 fine-tune 作业数有上限（常见约 6）；超限就睡 60s 重试
    while True:
        try:
            job = client.fine_tuning.jobs.create(
                training_file=train_file.id,
                validation_file=val_file.id,
                # 模型 ID / seed / 超参 / suffix 保持原文，影响训练结果与命名
                model="gpt-4.1-nano-2025-04-14",
                seed=42,
                hyperparameters={"n_epochs": 1, "batch_size": 1},
                suffix=f"pricer-{size}"
            )
            break
        except Exception as e:
            if "rate" in str(e).lower():
                print(f"  Size {size}: rate-limited, waiting 60s for a slot...")
                time.sleep(60)
            else:
                raise
    jobs[size] = job.id
    print(f"  Size {size}: job {job.id} launched")

print(f"\nAll {len(jobs)} jobs launched.")


## 监控作业

反复**重跑下一格**查看状态。七个作业大致同时间启动，也往往差不多同时结束。


In [ ]:
# ========== 轮询状态：全部 succeeded 才进入评估 ==========

all_done = True
for size, job_id in jobs.items():
    # retrieve：拉最新 job 对象（status / fine_tuned_model）
    status = client.fine_tuning.jobs.retrieve(job_id)
    # 训练中可能还没有 fine_tuned_model
    model_name = status.fine_tuned_model or "(training...)"
    print(f"  Size {size}: {status.status} -> {model_name}")
    if status.status != "succeeded":
        all_done = False

if all_done:
    print("\nAll jobs complete!")
else:
    print("\nStill training... re-run this cell in a few minutes.")


## 评估所有微调模型

先取出每个规模对应的 `fine_tuned_model` 名称，再在**同一 200 条测试集**上算 MAE，才能公平对比学习曲线。


In [ ]:
# ========== 收集微调后的模型 ID ==========

ft_models = {}
for size, job_id in jobs.items():
    info = client.fine_tuning.jobs.retrieve(job_id)
    # 记下 ft:... 模型名，供下一格 chat.completions 使用
    ft_models[size] = info.fine_tuned_model
    print(f"  Size {size}: {info.fine_tuned_model}")


In [ ]:
# ========== 对每个微调模型跑同一套 evaluate ==========

ft_results = {}

for size in SIZES:
    model_name = ft_models[size]
    print(f"\n--- Fine-tuned on {size} examples ({model_name}) ---")

    # 默认参数 _model=model_name：绑定当前循环的模型，避免闭包晚绑定踩坑
    def ft_predictor(item, _model=model_name):
        prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
        response = client.chat.completions.create(
            model=_model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=10,
            temperature=0
        )
        return response.choices[0].message.content

    preds, actuals, mae = evaluate(ft_predictor, test)
    # 存 preds/actuals 供后面散点图；mae 供学习曲线
    ft_results[size] = {"preds": preds, "actuals": actuals, "mae": mae}


## 结果

先看一张汇总表：随机 / 均值 / 零样本 / 各规模微调的 MAE 并排，谁赢一目了然。


In [ ]:
# ========== 汇总表：打印各方法 MAE ==========

print(f"{'Model':<25} {'MAE':>10}")
print("-" * 37)
print(f"{'Random baseline':<25} ${random_mae:>8.2f}")
print(f"{'Mean baseline':<25} ${mean_mae:>8.2f}")
print(f"{'Zero-shot GPT-4.1-nano':<25} ${zero_mae:>8.2f}")
# 按 SIZES 顺序打印每个微调规模
for size in SIZES:
    print(f"{'Fine-tuned (' + str(size) + ' ex.)':<25} ${ft_results[size]['mae']:>8.2f}")


### 学习曲线

横轴：训练样本数（0 = 零样本）；纵轴：MAE。看「加数据是否还在降误差」。


In [ ]:
# ========== 学习曲线：MAE 随训练规模变化 ==========

# x=0 放零样本点，再接各 SIZES
sizes_with_zero = [0] + SIZES
maes_curve = [zero_mae] + [ft_results[s]["mae"] for s in SIZES]

fig, ax = plt.subplots(figsize=(10, 5))
# 折线：蓝点连线
ax.plot(sizes_with_zero, maes_curve, "o-", color="#2563eb", linewidth=2, markersize=8)
# 水平虚线：零样本基线，方便看是否击败 0-shot
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, label=f"Zero-shot baseline (${zero_mae:.0f})")

# 每个点标注美元 MAE；点在基线上方则文字上移，否则下移避免重叠
for x, y in zip(sizes_with_zero, maes_curve):
    offset = 12 if y >= zero_mae else -18
    ax.annotate(f"${y:.0f}", (x, y), textcoords="offset points",
                xytext=(0, offset), ha="center", fontsize=10)

ax.set_xlabel("Training examples")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Learning Curve: How Much Training Data Does GPT-4.1-nano Need?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 所有方法的比较

条形图把基线、零样本、各规模微调放在同一纵坐标上；绿色柱表示该微调已打赢零样本。


In [ ]:
# ========== 条形图：所有方法并排比 MAE ==========

# 标签：Random / Mean / Zero-shot / FT-{size}
labels = ["Random", "Mean", "Zero-shot"] + [f"FT-{s}" for s in SIZES]
values = [random_mae, mean_mae, zero_mae] + [ft_results[s]["mae"] for s in SIZES]
# 灰=基线，橙=零样本；微调若优于零样本用绿，否则蓝
colors = ["#94a3b8", "#94a3b8", "#f59e0b"] + [
    "#10b981" if ft_results[s]["mae"] < zero_mae else "#2563eb" for s in SIZES
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors)
# 再画一条零样本参考线
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, alpha=0.7)

# 柱顶标注整美元 MAE
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"${val:.0f}", ha="center", fontsize=9)

ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("All Approaches Compared (green = beats zero-shot)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 预测价格 vs 实际价格

左图零样本、右图「MAE 最低」的微调规模；点越靠近对角线，定价越准。


In [ ]:
# ========== 散点图：零样本 vs 最佳微调 ==========

# 在 ft_results 里找 MAE 最小的训练规模
best_size = min(ft_results, key=lambda s: ft_results[s]["mae"])
best = ft_results[best_size]

# 1×2 子图：左边 0-shot，右边最佳 FT
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 统一坐标轴上限，两图可直接肉眼对比
max_price = max(max(zero_actuals), max(zero_preds),
                max(best["actuals"]), max(best["preds"]))

for ax, preds, actuals, title in [
    (ax1, zero_preds, zero_actuals, f"Zero-shot (MAE=${zero_mae:.0f})"),
    (ax2, best["preds"], best["actuals"], f"Fine-tuned {best_size} ex. (MAE=${best['mae']:.0f})"),
]:
    # x=真价，y=预测；理想情况落在 y=x 红虚线上
    ax.scatter(actuals, preds, alpha=0.4, s=15, color="#2563eb")
    ax.plot([0, max_price], [0, max_price], "--", color="#ef4444", linewidth=1)
    ax.set_xlabel("Actual price ($)")
    ax.set_ylabel("Predicted price ($)")
    ax.set_title(title)
    ax.set_xlim(0, max_price)
    ax.set_ylim(0, max_price)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
